In [32]:

!pip install pypdf2 arabic-reshaper python-bidi pyarabic pandas scikit-learn transformers torch -q

In [33]:
# Cellule 30 : Imports
import pdfplumber
import re
import pandas as pd
from bidi.algorithm import get_display
import pyarabic.araby as araby
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

print("✅ Toutes les bibliothèques sont chargées")

✅ Toutes les bibliothèques sont chargées


In [34]:
# Cellule 31 : Extraction du PDF
def extract_pdf_text(pdf_path):
    full_text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text:
                    full_text += get_display(text) + "\n"
                print(f"Page {i+1} traitée", end="\r")
        print(f"\n✅ {len(full_text)} caractères extraits")
        return full_text
    except Exception as e:
        print(f"❌ Erreur: {e}")
        return ""

# À MODIFIER : mettez le chemin correct de votre PDF
pdf_path = "C:/Users/hp/Desktop/code de la route MA52_05.pdf"
raw_text = extract_pdf_text(pdf_path)

Page 126 traitée
✅ 221996 caractères extraits


In [83]:
def normalize_arabic(text):
    if not text:
        return ""
    
    # ===== CORRECTION PRIORITAIRE =====
    # Remplacer tous les ء en début de mot par ا
    # Et corriger les patterns spécifiques
    
    # 1. Remplacer ءل par ال (le plus fréquent)
    text = text.replace('ءل', 'ال')
    
    # 2. Remplacer ء au début des mots par ا
    # Motif: ء + lettre (qui n'est pas l)
    text = re.sub(r'ء([^ل\s])', r'ا\1', text)
    
    # 3. Remplacer ء au milieu/fin par ا selon le contexte
    text = re.sub(r'([^\s])ء([^\s])', r'\1ا\2', text)
    
    # 4. Corrections spécifiques des mots fréquents
    corrections = {
        # Mots avec ء problématique
        'ءلحكام': 'أحكام',
        'ءلحكا': 'أحكام',
        'ءلحكام': 'أحكام',
        'ءجراء': 'إجراء',
        'ءعداد': 'إعداد',
        'ءن': 'أن',
        'ءو': 'أو',
        'ءي': 'أي',
        'ءه': 'أه',
        'ءم': 'أم',
        'ءب': 'أب',
        'ءت': 'أت',
        'ءث': 'أث',
        'ءج': 'أج',
        'ءح': 'أح',
        'ءخ': 'أخ',
        'ءد': 'أد',
        'ءذ': 'أذ',
        'ءر': 'أر',
        'ءز': 'أز',
        'ءس': 'أس',
        'ءش': 'أش',
        'ءص': 'أص',
        'ءض': 'أض',
        'ءط': 'أط',
        'ءظ': 'أظ',
        'ءع': 'أع',
        'ءغ': 'أغ',
        'ءف': 'أف',
        'ءق': 'أق',
        'ءك': 'أك',
        'ءل': 'ال',  # Déjà fait
        'ءلء': 'ألا',
        
        # Mots complets fréquents
        'املادة': 'المادة',
        'امل': 'الم',
        'اءل': 'ال',
        'اءن': 'أن',
        'اءو': 'أو',
        'اءي': 'أي',
        'اءه': 'أه',
        'اءم': 'أم',
        'اءب': 'أب',
        'اءت': 'أت',
        'اءث': 'أث',
        'اءج': 'أج',
        'اءح': 'أح',
        'اءخ': 'أخ',
        'اءد': 'أد',
        'اءذ': 'أذ',
        'اءر': 'أر',
        'اءز': 'أز',
        'اءس': 'أس',
        'اءش': 'أش',
        'اءص': 'أص',
        'اءض': 'أض',
        'اءط': 'أط',
        'اءظ': 'أظ',
        'اءع': 'أع',
        'اءغ': 'أغ',
        'اءف': 'أف',
        'اءق': 'أق',
        'اءك': 'أك',
        
        # Articles
        'ءلث': 'الث',
        'ءلج': 'الج',
        'ءلد': 'الد',
        'ءلر': 'الر',
        'ءلز': 'الز',
        'ءلس': 'الس',
        'ءلش': 'الش',
        'ءلص': 'الص',
        'ءلض': 'الض',
        'ءلط': 'الط',
        'ءلظ': 'الظ',
        'ءلع': 'الع',
        'ءلغ': 'الغ',
        'ءلف': 'الف',
        'ءلق': 'الق',
        'ءلك': 'الك',
        'ءلم': 'الم',
        'ءلن': 'الن',
        'ءله': 'اله',
        'ءلو': 'الو',
        'ءلي': 'الي',

        # ============================================================
        # AJOUT : corrections إ/أ sur les mots après remplacement ء→ا
        # Ces mots sont traités APRÈS les étapes 1-3 qui ont déjà
        # remplacé ء par ا, donc on corrige ici la hamza finale.
        # ============================================================

        # --- إ (hamza en-dessous) ---
        'الى'       : 'إلى',
        'الا'       : 'إلا',
        'اجراء'     : 'إجراء',
        'اجراءات'   : 'إجراءات',
        'اجراءه'    : 'إجراءه',
        'اجراءها'   : 'إجراءها',
        'اعداد'     : 'إعداد',
        'اعادة'     : 'إعادة',
        'الزامية'   : 'إلزامية',
        'الزام'     : 'إلزام',
        'الزامه'    : 'إلزامه',
        'الزامها'   : 'إلزامها',
        'الزاميا'   : 'إلزاميا',
        'الغاء'     : 'إلغاء',
        'الغاءه'    : 'إلغاءه',
        'الغاءها'   : 'إلغاءها',
        'الغاءهم'   : 'إلغاءهم',
        'ايداع'     : 'إيداع',
        'ايداعه'    : 'إيداعه',
        'ايداعها'   : 'إيداعها',
        'اخبار'     : 'إخبار',
        'اذن'       : 'إذن',
        'اذنا'      : 'إذنا',
        'ادانة'     : 'إدانة',
        'ادانات'    : 'إدانات',
        'ادارة'     : 'إدارة',
        'ادارية'    : 'إدارية',
        'اداريين'   : 'إداريين',
        'اداريا'    : 'إداريا',
        'اداري'     : 'إداري',
        'اضافة'     : 'إضافة',
        'اضافي'     : 'إضافي',
        'اضافية'    : 'إضافية',
        'احالة'     : 'إحالة',
        'احالته'    : 'إحالته',
        'احالتها'   : 'إحالتها',
        'احداث'     : 'إحداث',
        'احداثه'    : 'إحداثه',
        'اخضاع'     : 'إخضاع',
        'اخضاعه'    : 'إخضاعه',
        'اخضاعها'   : 'إخضاعها',
        'ادراج'     : 'إدراج',
        'ادراجه'    : 'إدراجه',
        'ادراجها'   : 'إدراجها',
        'اصابة'     : 'إصابة',
        'اصابات'    : 'إصابات',
        'اصالح'     : 'إصلاح',
        'اصالحات'   : 'إصلاحات',
        'اصالحه'    : 'إصلاحه',
        'اصالحها'   : 'إصلاحها',
        'اصدار'     : 'إصدار',
        'اصداره'    : 'إصداره',
        'اشعار'     : 'إشعار',
        'اشعاره'    : 'إشعاره',
        'اشعارهم'   : 'إشعارهم',
        'انتاج'     : 'إنتاج',
        'انهاء'     : 'إنهاء',
        'انذار'     : 'إنذار',
        'انجاز'     : 'إنجاز',
        'انجازه'    : 'إنجازه',
        'انجازها'   : 'إنجازها',
        'انارة'     : 'إنارة',
        'انشاء'     : 'إنشاء',
        'ايصال'     : 'إيصال',
        'ايصاله'    : 'إيصاله',
        'ايقاف'     : 'إيقاف',
        'ايقافه'    : 'إيقافه',
        'ايقافها'   : 'إيقافها',
        'افشاء'     : 'إفشاء',
        'افشاءها'   : 'إفشاءها',
        'افراغ'     : 'إفراغ',
        'افراغه'    : 'إفراغه',
        'افراغها'   : 'إفراغها',
        'اقامة'     : 'إقامة',
        'اقامته'    : 'إقامته',
        'اقامتهم'   : 'إقامتهم',
        'اقامتها'   : 'إقامتها',
        'اثبات'     : 'إثبات',
        'اثباته'    : 'إثباته',
        'اثباتها'   : 'إثباتها',
        'ارجاع'     : 'إرجاع',
        'ارجاعه'    : 'إرجاعه',
        'ارجاعها'   : 'إرجاعها',
        'ازالة'     : 'إزالة',
        'ازالته'    : 'إزالته',
        'ازالتها'   : 'إزالتها',
        'ازاحة'     : 'إزاحة',
        'ازاحته'    : 'إزاحته',
        'ازاحتها'   : 'إزاحتها',
        'اسقاط'     : 'إسقاط',
        'اتلاف'     : 'إتلاف',
        'اتلافه'    : 'إتلافه',
        'اتلافها'   : 'إتلافها',
        'اتمام'     : 'إتمام',
        'اجبارية'   : 'إجبارية',
        'اجبار'     : 'إجبار',
        'اجباري'    : 'إجباري',
        'اجباريا'   : 'إجباريا',
        'اطار'      : 'إطار',
        'اطاره'     : 'إطاره',
        'اطلاق'     : 'إطلاق',
        'اخفاء'     : 'إخفاء',
        'احدى'      : 'إحدى',
        'ادلاء'     : 'إدلاء',
        'ادلاءه'    : 'إدلاءه',

        # --- أ (hamza au-dessus) ---
        'احكام'     : 'أحكام',
        'احكامه'    : 'أحكامه',
        'احكامها'   : 'أحكامها',
        'احكامهم'   : 'أحكامهم',
        'اشغال'     : 'أشغال',
        'اطباء'     : 'أطباء',
        'اعضاء'     : 'أعضاء',
        'اعوان'     : 'أعوان',
        'اول'       : 'أول',
        'اولى'      : 'أولى',
        'اوىل'      : 'أولى',
        'اداء'      : 'أداء',
        'اداءها'    : 'أداءها',
        'ابعاد'     : 'أبعاد',
        'ابعادها'   : 'أبعادها',
        'اجهزة'     : 'أجهزة',
        'اجهزته'    : 'أجهزته',
        'اجهزتها'   : 'أجهزتها',
        'اجل'       : 'أجل',
        'اجله'      : 'أجله',
        'اجلها'     : 'أجلها',
        'اسباب'     : 'أسباب',
        'اسبابه'    : 'أسبابه',
        'اسبابها'   : 'أسبابها',
        'اشخاص'     : 'أشخاص',
        'اشخاصا'    : 'أشخاصا',
        'اشياء'     : 'أشياء',
        'اصحاب'     : 'أصحاب',
        'اصحابها'   : 'أصحابها',
        'اصناف'     : 'أصناف',
        'اصنافها'   : 'أصنافها',
        'اضرار'     : 'أضرار',
        'اضراره'    : 'أضراره',
        'اضرارها'   : 'أضرارها',
        'اطراف'     : 'أطراف',
        'اطرافها'   : 'أطرافها',
        'اعمال'     : 'أعمال',
        'اعمالها'   : 'أعمالها',
        'اغراض'     : 'أغراض',
        'اغراضه'    : 'أغراضه',
        'اغراضها'   : 'أغراضها',
        'افعال'     : 'أفعال',
        'افعاله'    : 'أفعاله',
        'اقصى'      : 'أقصى',
        'اقصاها'    : 'أقصاها',
        'اقل'       : 'أقل',
        'اكثر'      : 'أكثر',
        'الف'       : 'ألف',
        'الفين'     : 'ألفين',
        'الاف'      : 'آلاف',
        'امام'      : 'أمام',
        'امامه'     : 'أمامه',
        'امامها'    : 'أمامها',
        'امر'       : 'أمر',
        'امره'      : 'أمره',
        'امرها'     : 'أمرها',
        'امرهم'     : 'أمرهم',
        'امور'      : 'أمور',
        'اهلية'     : 'أهلية',
        'اهليته'    : 'أهليته',
        'اهليتها'   : 'أهليتها',
        'اهمية'     : 'أهمية',
        'ايام'      : 'أيام',
        'ايامها'    : 'أيامها',
        'ايضا'      : 'أيضا',
        'اربع'      : 'أربع',
        'اربعة'     : 'أربعة',
        'اربعين'    : 'أربعين',
        'احد'       : 'أحد',
        'احدهم'     : 'أحدهم',
        'اخرى'      : 'أخرى',
        'اخر'       : 'آخر',
        'اسطوانات'  : 'أسطوانات',
        'اسطنة'     : 'أسطنة',
    }
    
    for wrong, correct in corrections.items():
        text = text.replace(wrong, correct)
    
    # 5. Dernier passage : remplacer les ء restants par ا
    text = re.sub(r'ء', 'ا', text)
    
    # 6. Nettoyage pyarabic (si disponible)
    try:
        text = araby.strip_tashkeel(text)
        text = araby.normalize_hamza(text)
    except:
        pass

    # Ajouter un retour ligne SEULEMENT si "المادة" ressemble à un vrai début
    text = re.sub(
        r'(?<!\S)(المادة\s+[0-9٠-٩]+)',  # début mot (pas وسط phrase)
        r'\n\1',
        text
    )
    
    # 7. Nettoyage final
    #text = re.sub(r'\s+', ' ', text)

    # garder les retours ligne
    text = re.sub(r'[ \t]+', ' ', text)  # seulement espaces et tabs

    text = re.sub(r'[^\u0600-\u06FF0-9\s\.\،\;\:]', ' ', text)
    #text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[ \t]+', ' ', text).strip()
    
    return text
# Application
if raw_text:
    # 1. Normalisation
    normalized_text = normalize_arabic(raw_text)
    
    # 2. Extraction
    articles = extract_articles(normalized_text)
    
    # 3. Filtrage
    def is_real_article(text):
        return len(text) > 50

    articles = [a for a in articles if is_real_article(a)]

    print(f"✅ Articles : {len(articles)}")
    #print(f"✅ Normalisé : {len(normalized_text)} caractères")
    print("\n📄 Vérification des corrections:")
    print("-" * 50)
    print(normalized_text[:100967])

✅ Articles : 483

📄 Vérification des corrections:
--------------------------------------------------
القانون رقم 52.05 المتعلق
بمدونة السير على الطرق ،
كما وقع تغييره وتتميم ه
صيغة موطدة بتاري خ
10يوليو 2024
تم ءعداد هذه النسخة من ءجل تسهيل
مقروءية النص، وال يحتج ءال بالنصوص
في صيغتها المنشورة بالجريدة الرسمية
اءلمانة العامة للحكومة
القانون رقم 52.05
المتعلق بمدونة السير على الطرق الصادر بتنفيذه الظهير الشريف رقم 1.10.07
بتاريخ 26 من صفر 1431 11 فبراير 2010 ،
كما وقع تغييره وتتميمه
 ج.ر عدد 5824 بتاريخ 8 ربيع اءءلخر 1431 25 مارس 2010 ، ص : 2168 
الكتاب اءلول
شروط السير على الطريق العمومية
القسم اءلول
رخصة السياقة
الباب اءلول
ءلزامية رخصة السياقة

المادة 1
ال يجوز ءلي شخص ءن يسوق مركبة ذات محرك ءو مجموعة مركبات على الطريق العمومي ة
ما لم يكن حاصال على رخصة للسياقة سارية الصالحية ومسلمة من قبل اءلدارة، تناسب صنف
المركبة ءو مجموعة المركبات التي يسوقها.

المادة 2
استثناا من ءحكام المادة اءلولى ءعاله :
1 يجوز للمغاربة القاطنين بالخارج ءن يسوقوا، داخل التراب الوطني، خالل مدة ءقصاها
سنة وءحدة

In [86]:


import re

def extract_articles_grouped(text):
    pattern = r'(^المادة\s+([0-9٠-٩]+).*?)(?=^المادة\s+[0-9٠-٩]+|\Z)'
    
    matches = re.findall(pattern, text, re.DOTALL | re.MULTILINE)

    articles_dict = {}

    for full, num in matches:
        # convertir chiffres arabes → int
        num_clean = int(num.translate(str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')))

        if num_clean not in articles_dict:
            articles_dict[num_clean] = full.strip()
        else:
            #  fusionner les parties
            articles_dict[num_clean] += "\n" + full.strip()

    return articles_dict


articles_dict = extract_articles_grouped(normalized_text)

print(f"✅ Nombre réel d'articles : {len(articles_dict)}")

for num, content in list(articles_dict.items())[:147]:
    print(f"\n--- المادة {num} ---")
    print(content[:100000])

✅ Nombre réel d'articles : 318

--- المادة 1 ---
المادة 1
ال يجوز ءلي شخص ءن يسوق مركبة ذات محرك ءو مجموعة مركبات على الطريق العمومي ة
ما لم يكن حاصال على رخصة للسياقة سارية الصالحية ومسلمة من قبل اءلدارة، تناسب صنف
المركبة ءو مجموعة المركبات التي يسوقها.

--- المادة 2 ---
المادة 2
استثناا من ءحكام المادة اءلولى ءعاله :
1 يجوز للمغاربة القاطنين بالخارج ءن يسوقوا، داخل التراب الوطني، خالل مدة ءقصاها
سنة وءحدة ابتداا من ءقامتهم بالمغرب، بواسطة رخصة السياقة المسلمة لهم بالخارج سارية
الصالحية ؛
2 يجوز للساءقين من جنسية ءجنبية، ءن يسوقوا بواسطة رخصة السياقة المسلمة لهم بالخارج
سارية الصالحية، لكن فقط، خالل مدة ءقصاها سنة من تاريخ ءقامتهم المءقتة بالمغرب كما هي
محددة في النصوص التشريعية والتنظيمية المتعلقة بدخول وءقامة اءلجانب بالمملكة المغربية.
1
اءلمانة العامة للحكومة
المادة 2
ءعاله.

--- المادة 3 ---
المادة 3
يجب على الساءقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام المدة المشار ءليها
في المادة السابقة، ءن يتقدموا المتحانات الحصول على رخصة السياقة المغربية، ءو ءن يطلبوا تبديل
رخص

In [116]:
import re

# =========================
#  Nettoyage contenu
# =========================
def clean_article_content(content):
    # Supprimer les références de loi (numéros de type 52.05, 1.16.106...)
    content = re.sub(r'رقم\s+[\d\.]+', '', content)
    content = re.sub(r'\d+\.\d+\.\d+', '', content)
    # Supprimer les références à d'autres articles
    content = re.sub(r'(حسب|وفق|طبقا)\s+ل?لمادة\s+[0-9٠-٩]+.*?(?=\n|$)', '', content)
    # Supprimer les en-têtes répétés (الأمانة العامة للحكومة + numéros de page)
    content = re.sub(r'اءلمانة العامة للحكومة', '', content)
    content = re.sub(r'^\s*\d+\s*$', '', content, flags=re.MULTILINE)
    return content.strip()


# =========================
#  Extraction infos
# =========================
def extract_infos_rules(content):
    content = clean_article_content(content)

    # =========================
    #  AMENDE
    # =========================
    amende = None

    # Pattern 1 : "X درهم" ou "X إلى Y درهم"
    m = re.search(r'(\d{2,5})\s*(إلى|الى)?\s*(\d{2,5})?\s*(درهم|دراهم)', content)
    if m:
        # Prendre la valeur max trouvée
        vals = [int(x) for x in [m.group(1), m.group(3)] if x and x.isdigit()]
        vals = [v for v in vals if 100 <= v <= 50000]
        if vals:
            amende = max(vals)

    # Pattern 2 : tableau — ligne courte avec un nombre entre 200 et 15000
    if amende is None:
        for line in content.split('\n'):
            line = line.replace('O', '0').strip()
            if len(line) > 80:
                continue
            nums = re.findall(r'\b(\d{3,5})\b', line)
            candidates = [int(n) for n in nums if 200 <= int(n) <= 15000]
            if candidates:
                amende = max(candidates)
                break

    # =========================
    #  POINTS RETRAIT
    # =========================
    points = None

    # Pattern explicite : "خصم X نقطة/نقط"
    m = re.search(r'خصم\s+(\d+)\s+نق[طة]', content)
    if m:
        points = int(m.group(1))

    # Pattern tableau : petit nombre (1-30) sur ligne courte avec un grand nombre
    if points is None:
        for line in content.split('\n'):
            line = line.replace('O', '0').strip()
            if len(line) > 80:
                continue
            nums = re.findall(r'\b(\d{1,2})\b', line)
            pts = [int(n) for n in nums if 1 <= int(n) <= 30]
            # vérifier qu'il y a aussi une amende sur la même ligne
            has_amende = bool(re.search(r'\b\d{3,5}\b', line))
            if pts and has_amende:
                points = pts[0]
                break

    # =========================
    #  CATÉGORIES VÉHICULE
    # =========================
    categories = []

    # re.DOTALL pour matcher sur plusieurs lignes
    if re.search(r'دراجة.{0,20}نارية', content, re.DOTALL):
        categories.append('moto')
    if re.search(r'دراجة.{0,10}بمحرك', content, re.DOTALL):
        if 'moto' not in categories:
            categories.append('moto')
    if re.search(r'سيارة|مركبة.{0,30}خفيف', content, re.DOTALL):
        categories.append('voiture_legere')
    if re.search(r'(شاحنة|نقل البضاءع|نقل البضائع|وزن.*جمالي)', content, re.DOTALL):
        categories.append('poids_lourd')
    if re.search(r'نقل (الاشخاص|اءلشخاص|الأشخاص)', content, re.DOTALL):
        categories.append('transport_commun')

    if not categories:
        categories.append('non_precise')

    # =========================
    #  MOTS-CLÉS
    # =========================
    keywords = []
    kw_map = {
        'vitesse':        r'سرعة',
        'nuit':           r'ليل|ليلا',
        'alcool':         r'كحول|سكر|الكحول',
        'ceinture':       r'حزام',
        'telephone':      r'هاتف',
        'stationnement':  r'وقوف|توقف|ركن',
        # Attention : après normalisation, les hamzas sont souvent supprimées
        'priorite':       r'اولوية|أولوية|اءولوية',
        'signalisation':  r'اشارة|إشارة|تشوير|علامة',
        'permis':         r'رخصة السياقة',
        'alcootest':      r'كحول|نفخة|اختبار',
        'autoroute':      r'طريق سريع|اوتوروت',
        'feux':           r'ضوء|انارة|اءنارة',
        'depassement':    r'تجاوز',
        'arret':          r'توقف|وقوف',
    }

    for kw, pattern in kw_map.items():
        if re.search(pattern, content):
            keywords.append(kw)

    if not keywords:
        keywords.append('aucun')

    # =========================
    #  TYPE D'ARTICLE
    # =========================
    type_article = 'autre'
    if re.search(r'يعاقب|غرامة|درهم|خصم.*نق[طة]', content):
        type_article = 'sanction'
    elif re.search(r'يجب|يلتزم|يمنع|يحظر|لا يجوز', content):
        type_article = 'obligation'
    elif re.search(r'يراد|يقصد|تعريف|يعني', content):
        type_article = 'definition'

    return {
        'amende': amende,
        'points_retrait': points,
        'categorie_vehicule': list(set(categories)),
        'mots_cles': list(set(keywords)),
        'type_article': type_article,
    }


# =========================
#  APPLICATION
# =========================
results = []

if articles:
    iterable = articles.items() if isinstance(articles, dict) else enumerate(articles)

    for num, art in iterable:
        info = extract_infos_rules(art)
        results.append({
            "article": num,
            **info
        })

print(f"✅ Articles traités : {len(results)}\n")

# Afficher seulement les articles avec des infos utiles
utiles = [r for r in results if r['amende'] or r['points_retrait'] or r['mots_cles'] != ['aucun']]
print(f"📊 Articles avec données extraites : {len(utiles)}\n")
for r in utiles[:147]:
    print(r)

✅ Articles traités : 529

📊 Articles avec données extraites : 352

{'article': 1, 'amende': None, 'points_retrait': None, 'categorie_vehicule': ['non_precise'], 'mots_cles': ['permis'], 'type_article': 'autre'}
{'article': 2, 'amende': None, 'points_retrait': None, 'categorie_vehicule': ['non_precise'], 'mots_cles': ['permis'], 'type_article': 'obligation'}
{'article': 3, 'amende': None, 'points_retrait': None, 'categorie_vehicule': ['non_precise'], 'mots_cles': ['depassement'], 'type_article': 'autre'}
{'article': 5, 'amende': None, 'points_retrait': None, 'categorie_vehicule': ['non_precise'], 'mots_cles': ['alcool'], 'type_article': 'autre'}
{'article': 8, 'amende': None, 'points_retrait': None, 'categorie_vehicule': ['non_precise'], 'mots_cles': ['permis'], 'type_article': 'autre'}
{'article': 9, 'amende': 3500, 'points_retrait': None, 'categorie_vehicule': ['moto', 'transport_commun', 'poids_lourd'], 'mots_cles': ['permis', 'depassement'], 'type_article': 'autre'}
{'article': 10, 

In [121]:
import pandas as pd
import re

# =========================
#  Lignes parasites à ignorer dans la description
# =========================
NOISE_PATTERNS = [
    r'غريت\s+(ومتمت\s+)?مبوجب',
    r'ج\.ر\s+عدد',
    r'اءلمانة العامة للحكومة',
    r'^\s*\d+\s*$',
    r'بتاريخ\s+\d+\s+من\s+\w+\s+\d+',
    r'الصادر بتنفيذه الظهري',
]

def is_noise_line(line):
    for pat in NOISE_PATTERNS:
        if re.search(pat, line):
            return True
    return False

def clean_desc(raw_content, n_lines=3):
    lines = [l.strip() for l in raw_content.split('\n') if l.strip()]
    clean_lines = [l for l in lines if not is_noise_line(l)]
    return " ".join(clean_lines[:n_lines]) if clean_lines else ""

# =========================
#  Détection faux positifs amende
# =========================
def is_false_amende(value, content):
    if value is None:
        return False
    v = int(value)

    # Numéro de JO (ex: "ج.ر عدد 6490")
    if re.search(rf'ج\.ر\s+عدد\s+{v}\b', content):
        return True

    # Numéro de page isolé sur sa propre ligne
    if re.search(rf'^\s*{v}\s*$', content, re.MULTILINE):
        return True

    # Numéro d'article référencé SANS mention de درهم à proximité
    # → on ne supprime que si la valeur apparaît comme référence d'article
    #   ET qu'il n'y a pas de "درهم" dans les 60 caractères qui suivent le nombre
    ref_match = re.search(rf'[Mم]ادة\s+{v}\b', content)
    if ref_match:
        # Vérifier si ce même nombre apparaît aussi avec "درهم" quelque part
        has_dirham_context = bool(re.search(rf'\b{v}\b.{{0,60}}درهم', content))
        if not has_dirham_context:
            return True

    return False

# =========================
#  Construction du DataFrame
# =========================
rows = []

for r in results:
    art_num = r['article']

    if isinstance(articles, dict):
        raw_content = articles.get(art_num, "")
    else:
        raw_content = articles[art_num] if art_num < len(articles) else ""

    infraction_desc = clean_desc(raw_content, n_lines=3)

    amende = r.get("amende")
    if is_false_amende(amende, raw_content):
        amende = None

    rows.append({
        "article_id":         art_num,
        "infraction_desc":    infraction_desc,
        "type_article":       r.get("type_article", "autre"),
        "categorie_vehicule": ", ".join(r.get("categorie_vehicule", [])),
        "amende_fixe":        amende,
        "points_retrait":     r.get("points_retrait"),
        "mots_cles":          ", ".join(r.get("mots_cles", [])),
    })

df = pd.DataFrame(rows)

# =========================
#  Corriger article_id (extraire le vrai numéro depuis la description)
# =========================
def extract_real_article_num(row):
    m = re.search(r'المادة\s+(\d+)', row["infraction_desc"])
    if m:
        return int(m.group(1))
    return row["article_id"]

df["article_id"] = df.apply(extract_real_article_num, axis=1)

# =========================
# 🧹 Post-traitement numérique
# =========================
df["amende_fixe"]    = pd.to_numeric(df["amende_fixe"],    errors="coerce")
df["points_retrait"] = pd.to_numeric(df["points_retrait"], errors="coerce")

# Amendes réelles entre 200 et 20000 MAD
df.loc[~df["amende_fixe"].between(200, 20000), "amende_fixe"] = None

# =========================
# 🩹 Patch type_article : améliorer la détection "definition"
# =========================
def fix_type(row):
    art_num = row["article_id"]
    raw = articles.get(art_num, "") if isinstance(articles, dict) else (
        articles[art_num] if art_num < len(articles) else ""
    )
    if re.search(r'يقصد\s+ب|يراد\s+(في\s+مفهوم|ب)|تعريف|يعني\s+ب', raw):
        return 'definition'
    return row["type_article"]

df["type_article"] = df.apply(fix_type, axis=1)

# =========================
#  Colonnes dérivées
# =========================
df["has_sanction"] = df["type_article"] == "sanction"
df["has_amende"]   = df["amende_fixe"].notna()
df["has_points"]   = df["points_retrait"].notna()

# =========================
#  Aperçu & statistiques
# =========================
print(f"✅ DataFrame : {len(df)} lignes × {len(df.columns)} colonnes\n")

cols_show = ["article_id", "type_article", "amende_fixe", "points_retrait", "categorie_vehicule", "mots_cles"]
print("--- 10 articles avec amende ---")
print(df[df["has_amende"]].head(10)[cols_show].to_string(index=False))

print("\n--- Statistiques ---")
print(f"  Articles avec amende   : {df['has_amende'].sum()}")
print(f"  Articles avec points   : {df['has_points'].sum()}")
print(f"  Sanctions              : {df['has_sanction'].sum()}")
print(f"  Amendes min/max (MAD)  : {df['amende_fixe'].min()} / {df['amende_fixe'].max()}")
print(f"  Points min/max         : {df['points_retrait'].min()} / {df['points_retrait'].max()}")

print("\n--- Distribution type_article ---")
print(df["type_article"].value_counts())

print("\n--- Amendes < 300 MAD (à vérifier) ---")
suspects = df[df["amende_fixe"] < 300][["article_id", "amende_fixe", "infraction_desc"]]
print(suspects.to_string(index=False))

# =========================
#  Export CSV final
# =========================
output_path = "export_final.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n✅ Exporté : {output_path}")

✅ DataFrame : 529 lignes × 10 colonnes

--- 10 articles avec amende ---
 article_id type_article  amende_fixe  points_retrait                  categorie_vehicule           mots_cles
          7        autre       3500.0             NaN moto, transport_commun, poids_lourd permis, depassement
         40   obligation        500.0             3.0                         poids_lourd permis, depassement
         44   definition       1000.0             NaN   moto, voiture_legere, poids_lourd         depassement
         49   obligation        750.0             NaN                   moto, poids_lourd         depassement
         50   obligation        750.0             NaN                         poids_lourd         depassement
         53   obligation        750.0             NaN   moto, voiture_legere, poids_lourd         depassement
        118     sanction        500.0             NaN                         non_precise               aucun
        119     sanction       5000.0           